  <img src="../images/mackenzie_logo.jpg" width="25%" align="right"/>

# **PROJETO APLICADO IV - Ciência de Dados EaD - 2026/02**


# **Documentação do Projeto**

---

**Versão:** Etapa 3, entrega em 26/10/2026. O texto das Etapas 1 e 2 permanece. O incremento está na EDA, nos modelos e na primeira tabela de resultados. A base continua o snapshot com corte em 21/09/2026.


# **Paridade de preços de combustíveis, atividade econômica e a ação da Petrobras**
---


In [1]:
#@title **Identificação do Grupo**

#@markdown Integrantes do Grupo, nome completo em ordem alfabética (*informe: \<nome\>, \<matrícula\>*)
#@markdown AVA: grupo 19
Aluno1 = 'Guilherme Augusto Leal Oliveira, 10226108' #@param {type:"string"}
Aluno2 = 'Guilherme Rocha de Souza Duarte, 10730632' #@param {type:"string"}
Aluno3 = 'Ricardo Zulian de Souza Amaral, 10433651' #@param {type:"string"}
Aluno4 = '' #@param {type:"string"}
Aluno5 = '' #@param {type:"string"}


# **Resumo**

*(Correção — recorte PPI × volume, 09/10/2026.)*

O preço doméstico de diesel e gasolina no Brasil se afasta com frequência da paridade internacional. Esse desvio, medido pelo PPI da ABICOM e ponderado pelo volume de combustível que circulou (`prod`, produto desvio × volume, não produção), altera custo de frete e mobilidade e também o resultado financeiro e a cotação das ações da Petrobras. O estudo lê quatro alvos contra esse produto no mesmo período: o fechamento de PETR4, o preço ANP e o de lista da Petrobras, os indicadores de tráfego, indústria, varejo e desemprego, e o resultado trimestral de RT&M. A base é um snapshot congelado de séries oficiais (ABICOM, ANP, Petrobras, B3, ABCR, IBGE e BCB) em SQLite, com âncora diária no PPI de 5 de janeiro de 2022 a 21 de setembro de 2026. O modelo base é a linha ingênua e o SARIMAX com `prod` contemporâneo. No vetor diário entram também a LSTM e o TimesFM 3.0 com `prod`; o Prophet entra no preço e na atividade com o mesmo regressor. O produto apoia a leitura da política de preços e dos efeitos sobre trabalho, cidades e instituições (ODS 8, 11 e 16). Os resultados numéricos entram nas etapas 3 e 4.


# **Introdução**

*(Correção — recorte PPI × volume, 09/10/2026.)*

## Contexto

O Brasil combina preço na refinaria com a paridade internacional como referência de mercado. O PPI (Índice de Paridade de Preços) da ABICOM publica, em dias úteis, o diferencial em R＄/L e em % entre o preço Petrobras e a paridade de importação para gasolina e diesel (ABICOM, 2026). O diesel entra no frete; a gasolina, no tráfego leve das cidades. A Petrobras é empresa de economia mista listada na B3; o papel PETR4 reage a reajuste e à paridade. O tema é série temporal com frequências mistas: dia, semana, mês e trimestre.

O recorte liga três Objetivos de Desenvolvimento Sustentável. O ODS 8 trata de trabalho e crescimento: diesel, tráfego pesado, indústria e desemprego. O ODS 11 trata de cidades: gasolina, tráfego leve e preço ao consumidor. O ODS 16 trata de instituições: a política de preços de uma empresa estatal de capital aberto e misto e o PPI como sinal público do desvio em relação à paridade e ao interesse do acionista.

## Motivação

A política de preços da Petrobras mudou mais de uma vez nesta década. O mercado observa o preço internacional dos derivados e o PPI, mas a transmissão até o posto, o volume e as variáveis socioeconômicas não é uniforme. Fontes públicas (ANP, IBGE, ABCR, B3, BCB) publicam calendários distintos. O grupo junta essas séries em um recorte único a partir de 2022 e lê cada alvo contra o produto desvio × volume (`prod`) do mesmo período.

## Objetivo geral

O objetivo do projeto é medir o efeito do desvio de paridade ponderado pelo volume (`prod`) sobre PETR4, o preço de combustível, o tráfego, a indústria, o varejo, o desemprego e o resultado trimestral da Petrobras, e usar esse produto como previsório contemporâneo.

Objetivos específicos:

1. Organizar a base congelada com âncora no PPI diário.
2. Medir defasagens entre PPI, preço ANP/Petrobras e PETR4.
3. Testar se choques de diesel antecipam tráfego pesado, indústria e desemprego (ODS 8).
4. Testar se choques de gasolina e tráfego leve descrevem mobilidade urbana (ODS 11).
5. Documentar o PPI como sinal público da política de preços da Petrobras (ODS 16).
6. Entregar um notebook reproduzível com modelo estatístico de série temporal.

A hipótese de trabalho é que o choque de paridade e de preço passa primeiro pela ação e, depois, pela indústria, pelo varejo, pelo emprego e pelo tráfego.

## Justificativa

Um dos integrantes do grupo criou o pipeline ETL que captura e armazena esses dados. O pipeline está ativo. Nenhum estudo foi produzido a partir deste pipeline.

A contribuição analítica é um recorte reproduzível de fontes originais, com frequência mista e âncora diária, sem base pré-processada de terceiro. A contribuição prática é a leitura do `prod` contra preço, frete, mobilidade, emprego e resultado da companhia.

O ODS 8 liga diesel, emprego e crescimento. O ODS 11 liga gasolina, tráfego leve e cidades. O ODS 16 liga a transparência da política de preços à Petrobras como instituição.

O produto do curso é um experimento executável. Não é um painel de produção.

## Descrição da base de dados

A base do curso é o arquivo `data/gas_flare.sqlite`. O arquivo é um snapshot congelado. O projeto não acessa Postgres, API ao vivo nem credencial. As séries vêm de fontes originais (governo e entidades setoriais), reunidas em 14 tabelas. A âncora temporal é o PPI diário: 5 de janeiro de 2022 a 21 de setembro de 2026, com 1.151 datas. As demais famílias entram nesse calendário de dias úteis, cada uma com a defasagem da fonte.

Quadro 1 — Famílias da base congelada

| Família | Frequência | Conteúdo | Recorte no snapshot | Fonte |
|---|---|---|---|---|
| PPI | diária | diferencial Petrobras e independentes vs. paridade, gasolina e diesel, R＄/L e % | 5 jan. 2022 a 21 set. 2026 (1.151 datas) | ABICOM |
| Ação PETR4 | diária (pregão) | abertura, máxima, mínima, fechamento, volume | 1º fev. 2022 a 18 set. 2026 (1.156 pregões) | B3 |
| Preço ANP | semanal | média ao produtor/importador, linha Brasil: gasolina A comum; diesel S-10 e S-500, R＄/L | 3 jan. 2022 a 13 set. 2026 (245 semanas por produto) | ANP |
| Preço Petrobras | semanal irregular | preço de lista às distribuidoras, sem tributos, gasolina A e diesel S-10 e S-500, R＄/L | jan. 2022 a set. 2026 (64 semanas de gasolina; 72 de diesel) | Petrobras |
| Vendas ANP | mensal | volume nacional, m³ | jan. 2022 a jul. 2026 | ANP |
| Vendas e finanças Petrobras | trimestral | volume (mbpd), receita e margem RT&M | vendas 2022T1–2026T2; finanças de 2006T1 a 2026T2 | Petrobras |
| Tráfego | mensal | índice ABCR leve, pesado e total | fev. 2022 a ago. 2026 | ABCR |
| Atividade e emprego | mensal | desemprego (PNAD), varejo (PMC), indústria (PIM-PF) | jan. 2022 a jul. 2026 | IBGE |
| Inflação | mensal | IPCA, IPCA-15 e IGP-M | jan. 2022 a ago. 2026 | IBGE / FGV via BCB |

A coleta operacional de origem é semanal. O curso usa o estado congelado. Semanas sem PDF novo da Petrobras não avançam a série de preço à distribuidora: isso é ausência de reajuste, não falha de carga. PETR4 começa em fevereiro de 2022; os NaNs de janeiro no índice do PPI são esperados. Os preços da B3 não têm ajuste por proventos. O dicionário das tabelas está em `docs/metadados_base.md`.

### Atualização da base

A revisão da Etapa 2 conferiu a base congelada contra as fontes e encontrou erros de origem. O grupo levou esses erros ao pipeline que alimenta a base, e o pipeline corrigiu os pontos abaixo:

- PPI: leituras de OCR erradas, como −13,0 R＄/L em 26 de janeiro de 2023 e percentuais de três dígitos sem o primeiro algarismo em setembro de 2026.
- Preço ANP: o diesel misturava biodiesel B-100 e diesel marítimo ao diesel rodoviário, e a gasolina incluía a gasolina Premium. A carga também somava 22 cópias da linha Brasil como se fossem regiões.
- Preço Petrobras: algumas semanas vinham de tabelas de outro produto.
- Finanças Petrobras: os trimestres 1T26 e 2T26 estavam vazios.

Depois das correções, o grupo exportou uma nova base, com corte em 21 de setembro de 2026. O Quadro 1 descreve essa base. A versão anterior fica em `data/gas_flare_old.sqlite`, e a tabela `export_changelog` registra cada correção com a causa e a maior alteração de valor.


# **Referencial Teórico**

## Conceitos

Uma série temporal é um conjunto de observações coletadas em intervalo regular. Neste projeto, o PPI e o pregão de PETR4 seguem os dias úteis, enquanto preço, volume, tráfego e atividade chegam por semana, mês ou trimestre; reunidas no mesmo calendário, essas séries formam uma série multivariada.

A leitura de cada série separa quatro componentes: a tendência, que indica se a série sobe, fica estável ou cai; a sazonalidade, que se repete em período fixo, aqui o ano; o ciclo, que se repete sem período fixo; e o ruído, o movimento irregular que sobra depois dos outros três. Uma série estacionária mantém média e variância estáveis no tempo, o que não acontece quando há tendência, sazonalidade ou ciclo. A diferença entre valores consecutivos, ou a diferença sazonal, devolve a série a essa condição, e a função de autocorrelação (ACF) e a autocorrelação parcial (PACF) mostram quanto cada valor depende dos anteriores, o que orienta a ordem do modelo.

O preço de uma ação se comporta como série caótica, em que o movimento diário é quase todo ruído e não tem período. No recorte, o retorno diário de PETR4 tem média de 0,056% e desvio padrão de 2,1%, e por isso o grupo mantém curto o horizonte do vetor diário, de 5 e 21 dias úteis.

Dois tratamentos preparam as séries para os modelos. Um buraco no calendário recebe o valor anterior, o seguinte ou a média de uma janela curta, sem que o valor medido seja removido. Além disso, a criação de variáveis (*feature engineering*) transforma as séries brutas em indicadores: o projeto pondera o desvio do PPI pelo volume de combustível vendido e soma esse indicador no tempo para formar um estoque. Os modelos recebem esses indicadores prontos, como variáveis exógenas, e o cálculo está no Diagrama de Solução.

## Modelos

O ARIMA combina uma parte autorregressiva (AR), que usa valores passados, com uma média móvel (MA), que usa erros passados, e aplica a integração (I), isto é, a diferença, até a série ficar estacionária. O SARIMA acrescenta termos sazonais quando a série repete um padrão no ano, e o SARIMAX aceita ainda variáveis exógenas, como o PPI.

O Prophet segue outro caminho e trata a série como soma de uma tendência por partes, de sazonalidades em séries de Fourier, de feriados e de regressores externos (TAYLOR; LETHAM, 2018). Como o ajuste é uma regressão no tempo, o modelo não exige série estacionária, aceita datas faltantes e recebe o `prod` como regressor. Ele funciona bem em séries com tendência e ciclo anual, mas perde força em série caótica, como o preço diário de uma ação. Por isso o Prophet fica no preço semanal e na atividade mensal, e não no fechamento diário de PETR4.

A rede neural recorrente (RNN) devolve a saída do passo anterior à entrada e, assim, guarda memória da sequência. A LSTM (memória de longo e curto prazo) é a forma mais usada dessa rede. Neste recorte ela entra só no vetor diário, com dois canais: o fechamento de PETR4 e o `prod` contemporâneo do mesmo pregão.

O TimesFM é um modelo de fundação para séries temporais, desenvolvido pelo Google Research: um transformador do tipo decodificador que lê a série em trechos (*patches*) e prevê os trechos seguintes (DAS et al., 2024). A versão 3.0, de agosto de 2026, tem 330 milhões de parâmetros, foi pré-treinada com mais de um trilhão de pontos de séries temporais e aceita entrada com covariáveis. Neste recorte ele entra só no vetor diário, com PETR4 e o `prod` do treino como covariável passada. A API do TimesFM 3.0 não recebe regressor no horizonte. O código tem licença Apache 2.0, e os pesos da versão 3.0 têm licença não comercial, que permite o uso em estudo acadêmico.

## Trabalhos correlatos

Lopes et al. (2021) estudaram os fatores que moveram a ação PETR4 entre 2009 e 2020, com correlação de Pearson e regressão por mínimos quadrados ordinários. O preço do Brent, o Ibovespa e o volume de produção foram significativos, enquanto a política de preços de combustíveis apareceu só na discussão qualitativa. O trabalho explica o nível da ação, mas não faz previsão nem usa o PPI.

Puentes Márquez et al. (2021) previram a demanda de etanol combustível no Brasil e compararam o ARIMA com redes LSTM de uma e de várias camadas, em versão univariada e multivariada. A LSTM superou o ARIMA mesmo com 180 observações de treino. O trabalho não usa o PPI; no projeto a LSTM do vetor diário recebe o `prod` contemporâneo.

Serrano et al. (2025) previram o consumo de gasolina e de etanol no Brasil com séries de 72, 144 e 263 meses e compararam ARIMA/SARIMA, Holt-Winters, ETS, TBATS, Prophet, Orbit, N-BEATS e TFT. O TBATS teve o menor erro para a gasolina (MAPE de 3,22%) e o Orbit para o etanol (MAPE de 6,77%); como nenhum modelo venceu nos dois combustíveis, o estudo reforça a comparação de mais de uma técnica.

Os três trabalhos usam uma frequência só e não ponderam o PPI pelo volume. O projeto junta dia, semana, mês e trimestre e lê cada alvo contra o `prod` contemporâneo: naive e SARIMAX em todos os vetores; LSTM e TimesFM só no dia; Prophet no preço e na atividade.

# **Diagrama de Solução**

*(Incremento da Etapa 3 — 26/10/2026.)*

O pipeline segue as cinco fases da análise de séries temporais: coleta, pré-processamento, análise exploratória, modelagem e avaliação. O produto desvio × volume diário e o estoque são calculados antes dos modelos, que recebem esses indicadores prontos. Cada alvo entra contra o `prod` (ou `prod_pbr`) do mesmo período. A linha ingênua é a dummy da disciplina; o SARIMAX é o modelo base. A LSTM e o TimesFM 3.0 entram só no vetor diário, com `prod`. O Prophet entra no preço e na atividade com o mesmo regressor. O trimestre fica com naive e SARIMAX (`prod_pbr`).

```
1. COLETA
   base congelada data/gas_flare.sqlite
   âncora: PPI diário (ABICOM), 1.151 datas, corte em 21/09/2026
        |
        v
2. PRÉ-PROCESSAMENTO
   alinhamento ao calendário de dias úteis
   buraco de calendário -> valor vizinho
   conferência da leitura OCR do PPI
   volume diário (ANP: m³ do mês; Petrobras: mil barris por dia)
   produto do dia = desvio em R＄/L × volume diário
   soma dos produtos = estoque
        |
        v
3. ANÁLISE EXPLORATÓRIA E VISUALIZAÇÃO
   tendência, sazonalidade, ciclo e ruído
   estacionariedade e diferença
   ACF e PACF
   correlação cruzada PPI × alvo -> defasagem
   gráfico de linha, de caixa e mapa de calor
        |
        v
4. MODELAGEM (um corte temporal por vetor)
        +-------------+-------------+-------------+
        |             |             |             |
        v             v             v             v
     mercado        preço       atividade     companhia
     dia útil       semana         mês        trimestre
     naive          naive        naive         naive
     SARIMAX        SARIMAX      SARIMAX       SARIMAX
     LSTM           Prophet      Prophet       prod. próprio trim.
     TimesFM 3.0
        |             |             |             |
        +-------------+------+------+-------------+
                             |
                             v
5. AVALIAÇÃO E VALIDAÇÃO
   erro absoluto médio, erro quadrático médio e sua raiz
   gráfico da série real contra a prevista
   horizonte próprio de cada vetor
```

## Pré-processamento

**PPI.** O cartão da ABICOM traz o desvio em R＄/L e em %, e o percentual é calculado sobre o preço vigente da Petrobras: (preço Petrobras − paridade) / preço Petrobras. A razão entre as duas colunas fica constante entre reajustes, o que confirma essa conta. Em 5 de maio de 2026, por exemplo, a gasolina marca −R＄ 2,22/L e −88%, com o preço Petrobras perto de R＄ 2,52/L e a paridade perto de R＄ 4,74/L; o reajuste até a paridade seria de 88% sobre o preço vigente. Em 2026, o preço Petrobras usado pelo PPI é o preço com desconto das tabelas da Petrobras, enquanto essa coluna existe, ao passo que a série de preço às distribuidoras usa o preço de lista, e por isso as duas séries diferem nesse ano.

**Leitura OCR.** Os valores do PPI saem da imagem da ABICOM por OCR. A revisão da Etapa 2 encontrou leituras erradas, que o pipeline de origem corrigiu contra a imagem, e uma regra passou a comparar cada percentual com o preço de referência dos dez dias anteriores. Quando a divergência passa de 1,5 ponto percentual, a linha vai para conferência manual, marcada na coluna `man_insp`.

**Volume diário.** O desvio em R＄/L entra multiplicado por um volume diário, porque a soma do desvio sem volume não mede o combustível que circulou. Como a base não publica venda diária, os vetores de mercado, preço e atividade tiram o volume diário da venda mensal da ANP, com o volume em m³ vezes 1.000, dividido pelos dias do mês. A ANP soma todos os fornecedores, e a Petrobras responde por cerca de 58% desse volume no 3º trimestre de 2025; por isso o vetor da companhia usa a taxa da própria Petrobras, em mil barris por dia, vezes 159 L por barril. O produto do dia é o desvio vezes esse volume, em R＄, e a série acumulada soma esses produtos. O percentual fica fora da multiplicação, porque a divisão pelo preço já ocorreu. A exógena é sempre do mesmo período do alvo: não há defasagem de PPI × volume.

**Escala e calendário.** A LSTM recebe PETR4 e `prod` normalizados entre 0 e 1 no treino. O SARIMAX, o Prophet e o TimesFM recebem os valores originais. Como o pregão de PETR4 começa em 1º de fevereiro de 2022, as datas de janeiro de 2022 saem do treino do vetor diário.

## Análise exploratória

A leitura de cada vetor separa tendência, sazonalidade, ciclo e ruído, testa a estacionariedade e mede a ACF e a PACF. A EDA preliminar já mostra três padrões. A venda de diesel da ANP tem pico no 3º trimestre em 2022 e em 2023, e a participação da Petrobras no volume da ANP cai cerca de 0,9 ponto percentual por trimestre desde 2022. Em 2026, a variação semanal típica do preço ANP é três a quatro vezes maior que em 2022–2025; como esse preço é média de produtores e importadores e o preço Petrobras fica muito abaixo da paridade nesse ano, uma causa provável é a parcela importada, que muda de semana para semana. Os gráficos, os testes ADF e a correlação cruzada de cada vetor ficam na seção EDA desta etapa.

A correlação cruzada entre o PPI e cada alvo mede em quantos dias, semanas ou meses o alvo responde ao choque. Essa leitura fica na EDA. No modelo, o PPI e o produto desvio × volume entram no **mesmo** período do alvo, sem defasagem.

## Modelagem

Como a base junta dia, semana, mês e trimestre, a proposta separa quatro vetores, para não levar o horizonte de 5 e 21 dias úteis às séries mais lentas.

| Vetor | Alvo | Entrada | Horizonte | Modelo |
|---|---|---|---|---|
| Mercado | fechamento de PETR4 | `prod` do mesmo dia | 5 e 21 dias úteis | naive, SARIMAX, LSTM e TimesFM 3.0, todos com `prod` (naive sem `prod`) |
| Preço | preço da ANP e preço da Petrobras | `prod` da mesma semana | 4 e 8 semanas | naive, SARIMAX sazonal e Prophet, com `prod` |
| Atividade | tráfego pesado, indústria, varejo e desemprego | `prod` somado no mês | 1 e 3 meses | naive, SARIMAX sazonal e Prophet, com `prod` |
| Companhia | volume, receita RT&M, lucro bruto e margem | `prod_pbr` do próprio trimestre | 2º trimestre de 2026 | naive e SARIMAX com `prod_pbr`, conferido com o ITR publicado |

O modelo base desta etapa é a linha ingênua e o SARIMAX com ordem de `auto_arima`. O Prophet entra no preço e na atividade com o mesmo `prod` contemporâneo (`add_regressor`). A LSTM e o TimesFM 3.0 entram só no vetor diário, com `prod`. O teste usa o produto observado nesse horizonte, salvo o TimesFM, cuja API só lê `prod` no treino. O termo sazonal entra quando a EDA mostra repetição anual. No vetor da companhia o SARIMAX recebe o `prod_pbr` daquele trimestre e força `d=1`. O 1T25 usa a soma de 1º de janeiro a 31 de março de 2025; o 2T26 usa abril a junho de 2026. A série mensal, com cerca de 55 pontos, fica com o SARIMAX e o Prophet; a trimestral, com 17, fica só com o SARIMAX.

## Avaliação e partição

Em todos os vetores, a avaliação usa o erro absoluto médio, o erro quadrático médio, a raiz desse erro e o gráfico da série real contra a prevista.

A partição usa só a base congelada `data/gas_flare.sqlite`, com corte em 21 de setembro de 2026, sem nenhuma observação nova. O corte é temporal e há um por vetor; no vetor diário, ele é o mesmo para o SARIMAX, a LSTM e o TimesFM 3.0. O teste tem o tamanho do horizonte mais longo, e o horizonte curto é medido dentro desse mesmo teste: 5 dias úteis dentro dos 21, 4 semanas dentro das 8 e 1 mês dentro dos 3.

Alguns pontos ficam de fora para manter as séries alinhadas. O PPI de 21 de setembro de 2026 não entra, porque o pregão de PETR4 termina em 18 de setembro, e o tráfego de agosto de 2026 também não, porque as outras séries mensais param em julho. No resultado da Petrobras, o teste é o 2º trimestre de 2026, último trimestre publicado, cuja receita inclui R＄ 9,7 bilhões de subvenção ao diesel e R＄ 0,8 bilhão à gasolina, fato sem paralelo no treino. Como esse teste tem um ponto só, o erro desse vetor serve apenas como indicação.

| Vetor | Treino | Teste |
|---|---|---|
| Mercado | 1º fev. 2022 a 19 ago. 2026 | 20 ago. 2026 a 18 set. 2026 (21 dias úteis) |
| Preço | até a semana encerrada em 19 jul. 2026 | 20 jul. 2026 a 13 set. 2026 (8 semanas) |
| Atividade | até abr. 2026 | mai., jun. e jul. 2026 |
| Companhia | 1º trim. 2022 ao 1º trim. 2026 | 2º trim. 2026, resultado já publicado |

Os números parciais e os gráficos real × previsto ficam na seção Resultados. A discussão do melhor modelo e da defasagem do PPI fica na Etapa 4.


# **Cronograma**

*(Incremento da Etapa 3 — 26/10/2026.)*

Até 9 de outubro de 2026 o grupo concluiu a base congelada com âncora no PPI (objetivo específico 1), mediu as defasagens no treino (objetivo 2) e alinhou o recorte: naive e SARIMAX em todos os vetores, LSTM e TimesFM com `prod` só no dia, Prophet com `prod` no preço e na atividade (objetivo 6). A passagem completa no MacBook atualiza o JSON. A tabela abaixo traz só o que resta, com origem nos objetivos específicos e divisão entre os integrantes.

| Tarefa | Obj. | Responsável | Início | Término |
|---|---|---|---|---|
| Executar o notebook, gravar as saídas das células e publicar o GitHub | 6 | Ricardo | 06/10/2026 | 12/10/2026 |
| Aplicar o feedback da Etapa 2 quando a nota sair (siglas, citações, template) | 6 | grupo | 13/10/2026 | 19/10/2026 |
| Conferir figuras numeradas, pipeline e entrega da Etapa 3 | 6 | Guilherme L. | 20/10/2026 | 26/10/2026 |
| Comparar as famílias nas tabelas e nos gráficos real × previsto | 6 | Guilherme D. | 27/10/2026 | 02/11/2026 |
| Analisar a defasagem do PPI frente ao erro de previsão (ODS 8 e 11) | 2, 3, 4 | Ricardo | 03/11/2026 | 09/11/2026 |
| Redigir Discussão e Conclusão; Encontro 4 em 10/11, 19h | 6 | Ricardo | 10/11/2026 | 16/11/2026 |
| Preparar slides e roteiro do vídeo (5 a 10 min) | 5, 6 | Guilherme L. | 17/11/2026 | 23/11/2026 |
| Gravar, publicar o vídeo e revisar o notebook final | 6 | grupo | 24/11/2026 | 30/11/2026 |

*Os dias 26/10 e 30/11 são marcos do curso. O Encontro 4 foi marcado para 10/11/2026, às 19h.*


# **EDA e Pré-processamento dos dados**

*(Incremento da Etapa 3 — 26/10/2026.)*

A base congelada `data/gas_flare.sqlite` alimenta os quatro vetores com o corte de 21 de setembro de 2026. O recorte diário segue o calendário de PETR4, de 1º de fevereiro de 2022 a 18 de setembro de 2026 (1.156 pregões), e o PPI de 21 de setembro fica de fora porque não há fechamento nesse dia. O volume diário dos vetores de mercado, preço e atividade sai da venda mensal da ANP: o m³ do mês vezes 1.000, dividido pelos dias do mês; no vetor da companhia, a taxa da Petrobras em mil barris por dia vira litros com o fator 1.000 × 159 L. O produto do dia é o desvio em R＄/L vezes esse volume, e o estoque soma esses produtos. Como a ANP para em julho de 2026, agosto e setembro repetem a última taxa conhecida, e qualquer buraco de calendário recebe o valor anterior.

Os testes de estacionariedade, a ACF, a PACF e a correlação cruzada usam só a janela de treino de cada vetor. A LSTM recebe PETR4 e `prod` em escala 0–1 ajustada nesse treino. As Figuras 1 a 13 e o Quadro 3 descrevem cada vetor; o recorte de treino e teste está no Quadro 2.

Quadro 2 — Recortes de treino e teste

| Vetor | Treino | Teste | Horizontes |
|---|---|---|---|
| Mercado | 1º fev. 2022 a 19 ago. 2026 | 20 ago. 2026 a 18 set. 2026 (21 dias úteis) | 5 e 21 dias |
| Preço | até a semana encerrada em 19 jul. 2026 | 20 jul. 2026 a 13 set. 2026 (8 semanas) | 4 e 8 semanas |
| Atividade | até abr. 2026 | mai., jun. e jul. 2026 | 1 e 3 meses |
| Companhia | 1º trim. 2022 ao 1º trim. 2026 | 2º trim. 2026 | um trimestre |


In [2]:
# Ambiente e leitura da base congelada
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent
sys.path.insert(0, str(project_root / "src"))

from db_access import read_engine, resolve_database_url
from paths import (
    ensure_output_dirs,
    figures_eda_dir,
    figures_forecasts_dir,
    images_dir,
    results_dir,
)

ENGINE = read_engine(project_root)
ensure_output_dirs(project_root)
IMAGES = images_dir(project_root)
FIGURES_EDA = figures_eda_dir(project_root)
FIGURES_FC = figures_forecasts_dir(project_root)
RESULTS = results_dir(project_root)
print(resolve_database_url(project_root))
device = "cpu"


sqlite:///D:\Users\Ricardo\Documents\opencode\proj_IV\data\gas_flare.sqlite


In [3]:
def sql(q: str) -> pd.DataFrame:
    return pd.read_sql_query(q, ENGINE)

ppi = sql(
    "SELECT report_date, pbr_gas_R, pbr_gas_pct, pbr_die_R, pbr_die_pct "
    "FROM ppi_daily ORDER BY report_date"
)
ppi["report_date"] = pd.to_datetime(ppi["report_date"])
ppi = ppi.set_index("report_date").sort_index()

eq = sql("SELECT tradedate, price_last FROM equity_daily WHERE ticker='PETR4' ORDER BY tradedate")
eq["tradedate"] = pd.to_datetime(eq["tradedate"])
eq = eq.set_index("tradedate").sort_index()["price_last"]

anp_sales = sql("SELECT product, month, volume_m3 FROM anp_fuel_sales ORDER BY month, product")
anp_sales["month"] = pd.to_datetime(anp_sales["month"])
anp_wide = anp_sales.pivot(index="month", columns="product", values="volume_m3")
anp_wide.columns = [f"anp_{c}_m3" for c in anp_wide.columns]

def weekly(table, col):
    d = sql(f"SELECT week_start, avg_price_rs_l AS {col} FROM {table}")
    d["week_start"] = pd.to_datetime(d["week_start"])
    return d.set_index("week_start").sort_index()

price_week = (
    weekly("anp_distributor_weekly_gasoline", "anp_gas")
    .join(weekly("anp_distributor_weekly_diesel", "anp_die"), how="outer")
    .join(weekly("petrobras_distributor_weekly_gasoline", "pbr_gas"), how="outer")
    .join(weekly("petrobras_distributor_weekly_diesel", "pbr_die"), how="outer")
    .sort_index()
)
PRICE_TEST_END = pd.Timestamp("2026-09-13")
week_idx = pd.date_range(price_week.index.min(), PRICE_TEST_END, freq="W-MON")
price_week = price_week.reindex(week_idx).ffill()

abcr = sql("SELECT month, vehicle_type, traffic_index FROM abcr_traffic_index")
abcr["month"] = pd.to_datetime(abcr["month"])
abcr_wide = abcr.pivot(index="month", columns="vehicle_type", values="traffic_index").sort_index()
abcr_wide = abcr_wide.rename(columns={"light": "abcr_light", "heavy": "abcr_heavy", "total": "abcr_total"})

ibge = sql("SELECT indicator_code, month, value FROM ibge_indicators")
ibge["month"] = pd.to_datetime(ibge["month"])
ibge_wide = ibge.pivot(index="month", columns="indicator_code", values="value").sort_index()
ibge_wide = ibge_wide.rename(columns={"6381": "unemp", "8880": "pmc", "8888": "pim"})

fin = sql(
    "SELECT year, quarter_num, diesel_mbpd, gasoline_mbpd, "
    "diesel_revenue_rsmn, gasoline_revenue_rsmn, "
    "rtm_sales_revenue_rsmn, rtm_gross_profit_rsmn, rtm_gross_margin_pct, "
    "extraction_note FROM pbr_quarterly_diesel_gasoline "
    "WHERE year >= 2022 ORDER BY year, quarter_num"
)
fin["quarter_end"] = pd.to_datetime(
    fin.apply(lambda r: pd.Timestamp(int(r.year), int(r.quarter_num) * 3, 1) + pd.offsets.MonthEnd(0), axis=1)
)
fin = fin.set_index("quarter_end").sort_index()

print("ppi", len(ppi), ppi.index.min().date(), ppi.index.max().date())
print("petr4", eq.notna().sum(), eq.index.min().date(), eq.index.max().date())
print("anp meses", anp_wide.dropna(how="all").shape[0], anp_wide.index.min().date(), anp_wide.index.max().date())
print("preço semanas", len(price_week))
print("fin trimestres 2022+", len(fin))


ppi 1151 2022-01-05 2026-09-21
petr4 1156 2022-02-01 2026-09-18
anp meses 55 2022-01-01 2026-07-01
preço semanas 245
fin trimestres 2022+ 18


In [4]:
LITERS_PER_M3 = 1000.0
LITERS_PER_MBPD = 1000.0 * 159.0  # mil barris/dia → L/dia

MKT_TRAIN_END = pd.Timestamp("2026-08-19")
MKT_TEST_END = pd.Timestamp("2026-09-18")
PRICE_TRAIN_END = pd.Timestamp("2026-07-19")
ACT_TRAIN_END = pd.Timestamp("2026-04-01")
ACT_TEST_END = pd.Timestamp("2026-07-01")

daily_idx = eq.index
vol = pd.DataFrame(index=daily_idx)
vol["month"] = vol.index.to_period("M").to_timestamp()
vol = vol.join(anp_wide, on="month")
vol["anp_gasolina_c_m3"] = vol["anp_gasolina_c_m3"].fillna(anp_wide["anp_gasolina_c_m3"].dropna().iloc[-1])
vol["anp_diesel_m3"] = vol["anp_diesel_m3"].fillna(anp_wide["anp_diesel_m3"].dropna().iloc[-1])
days = vol.index.to_series().dt.days_in_month
vol["vol_gas_l"] = vol["anp_gasolina_c_m3"] * LITERS_PER_M3 / days
vol["vol_die_l"] = vol["anp_diesel_m3"] * LITERS_PER_M3 / days

pbr_daily = pd.DataFrame(index=daily_idx)
pbr_daily["qend"] = pd.DatetimeIndex(pbr_daily.index.to_period("Q").end_time).normalize()
pbr_daily = pbr_daily.join(fin[["diesel_mbpd", "gasoline_mbpd"]], on="qend")
pbr_daily[["diesel_mbpd", "gasoline_mbpd"]] = pbr_daily[["diesel_mbpd", "gasoline_mbpd"]].ffill()

daily = pd.DataFrame({"petr4": eq}, index=daily_idx)
daily = daily.join(ppi[["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]])
daily[["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]] = daily[
    ["pbr_gas_R", "pbr_die_R", "pbr_gas_pct", "pbr_die_pct"]
].ffill()
daily["vol_gas_l"] = vol["vol_gas_l"]
daily["vol_die_l"] = vol["vol_die_l"]
daily["prod"] = daily["pbr_gas_R"] * daily["vol_gas_l"] + daily["pbr_die_R"] * daily["vol_die_l"]
daily["stock"] = daily["prod"].cumsum()
daily["prod_pbr"] = (
    daily["pbr_gas_R"] * pbr_daily["gasoline_mbpd"] * LITERS_PER_MBPD
    + daily["pbr_die_R"] * pbr_daily["diesel_mbpd"] * LITERS_PER_MBPD
)
daily["stock_pbr"] = daily["prod_pbr"].cumsum()

mkt_train = daily.loc[:MKT_TRAIN_END]
price_train = price_week.loc[:PRICE_TRAIN_END]
print("treino diário", mkt_train.index.min().date(), "→", mkt_train.index.max().date(), "n=", len(mkt_train))
print("produto diário (média R$ mi)", round(mkt_train["prod"].mean() / 1e6, 1))


treino diário 2022-02-01 → 2026-08-19 n= 1135
produto diário (média R$ mi) -104.2


In [5]:
import sys
import warnings
warnings.filterwarnings("ignore")

if sys.platform == "win32":
    print("Neste Windows o ADF do statsmodels não completa (numpy.linalg).")
    print("Os números do treino estão no texto. No MacBook M4 esta célula reproduz o ADF.")
else:
    from statsmodels.tsa.stattools import adfuller, ccf

    def adf_row(name, s):
        x = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
        if len(x) < 20:
            return {"série": name, "n": int(len(x)), "ADF": np.nan, "p": np.nan, "lags": np.nan}
        stat, p, lags, *_ = adfuller(x.to_numpy(dtype=float), autolag="AIC")
        return {"série": name, "n": int(len(x)), "ADF": round(float(stat), 3), "p": round(float(p), 4), "lags": int(lags)}

    def best_ccf(x, y, max_lag):
        aligned = pd.concat([x, y], axis=1).dropna()
        vals = ccf(aligned.iloc[:, 0], aligned.iloc[:, 1], adjusted=False)[: max_lag + 1]
        k = int(np.argmax(np.abs(vals)))
        return k, float(vals[k])

    adf_daily = pd.DataFrame([
        adf_row("PETR4", mkt_train["petr4"]),
        adf_row("PETR4 d1", mkt_train["petr4"].diff()),
        adf_row("PPI gasolina R$/L", mkt_train["pbr_gas_R"]),
        adf_row("PPI diesel R$/L", mkt_train["pbr_die_R"]),
        adf_row("produto do dia", mkt_train["prod"]),
        adf_row("produto do dia d1", mkt_train["prod"].diff()),
    ])
    display(adf_daily)

    print("CCF PETR4 ← PPI (treino, 0–21 pregões)")
    for col, lab in [("pbr_gas_R", "gasolina R$"), ("pbr_die_R", "diesel R$"), ("prod", "produto")]:
        k, v = best_ccf(mkt_train[col], mkt_train["petr4"], 21)
        print(f"  {lab}: defasagem {k}, ccf {v:.3f}")


Neste Windows o ADF do statsmodels não completa (numpy.linalg).
Os números do treino estão no texto. No MacBook M4 esta célula reproduz o ADF.


## Vetor de mercado

O fechamento de PETR4 no treino (1.135 pregões até 19 de agosto de 2026) não é estacionário (ADF −1,86), mas a primeira diferença é (ADF −32,9). O retorno diário nesse treino tem média 0,046% e desvio-padrão 2,1%, o que mantém curtos os horizontes de 5 e 21 dias úteis.

A Figura 1 percorre os 1.156 pregões até 18 de setembro de 2026. A série sobe no recorte e oscila sem um ciclo anual visível, o que combina com o ADF no nível e com a ausência de sazonalidade semanal no SARIMAX diário.

![Fechamento de PETR4](../figures/eda/eda_mercado_petr4.svg)

*Figura 1 — Fechamento de PETR4, 1º de fevereiro de 2022 a 18 de setembro de 2026.*

A Figura 2 mostra a ACF e a PACF do treino. A ACF do nível decai devagar, padrão de raiz unitária, e a PACF da primeira diferença corta cedo. Esse desenho leva o `auto_arima` a (0,1,0): o próximo fechamento é o último mais um ruído, sem termo AR ou MA longo.

![ACF e PACF de PETR4 no treino](../figures/eda/eda_mercado_acf.svg)

*Figura 2 — ACF e PACF de PETR4 no treino (até 19 de agosto de 2026).*

O desvio do PPI em R＄/L já é estacionário no treino (gasolina ADF −3,49; diesel −3,53). O produto diário desvio × volume ANP ainda traz tendência (ADF −2,11) e estaciona na diferença (ADF −9,43). A correlação cruzada com PETR4 é contemporânea e negativa: −0,55 com o PPI da gasolina, −0,57 com o do diesel e −0,59 com o produto do dia, todas no lag 0. O modelo usa esse produto no mesmo dia do fechamento, inclusive no teste.

A Figura 3 registra o desvio em R＄/L, já estacionário, com o aprofundamento de 2026, quando o cartão da ABICOM usa o preço Petrobras com desconto. A Figura 4 isola o produto diário, que ainda carrega a tendência do volume, e a Figura 5 acumula esses produtos no estoque. O produto entra no modelo como exógena já calculada, e o estoque permanece descritivo.

![Desvio PPI em R＄/L](../figures/eda/eda_ppi_desvio.svg)

*Figura 3 — Desvio do PPI em R＄/L, gasolina e diesel, no recorte diário.*

![Produto diário desvio × volume](../figures/eda/eda_indicador_prod.svg)

*Figura 4 — Produto diário desvio × volume ANP.*

![Estoque acumulado](../figures/eda/eda_indicador_stock.svg)

*Figura 5 — Estoque acumulado do produto diário.*


## Vetor de preço

As séries semanais de preço ANP no treino (237 semanas) não são estacionárias no nível (gasolina ADF −2,58; diesel −2,01) e estacionam na primeira diferença. O preço de lista da Petrobras, reindexado às segundas e preenchido para a frente nas semanas sem PDF novo, também fica no nível (gasolina −2,65; diesel −2,25). A ACF do diesel ANP no treino decai devagar, o que pede termo sazonal anual (`s=52`) no SARIMAX e no Prophet.

A Figura 6 junta as quatro séries semanais. O preço ANP se move semana a semana, e o preço de lista da Petrobras avança em degraus, parado nas semanas sem PDF novo. A linha ingênua cola nesse degrau. O SARIMAX e o Prophet recebem o `prod` da semana e, no teste, reagem ao produto contemporâneo. A Figura 7 traz a ACF e a PACF do diesel ANP: a ACF lenta no nível confirma a diferença, e o período de 52 semanas justifica o termo sazonal, ainda que a busca `auto_arima` tenha devolvido (0,1,0)(0,0,0)[52] para o diesel.

A correlação cruzada com o PPI da semana aponta máximo em defasagem 1 na gasolina ANP (ccf −0,57) e defasagem 2 no diesel ANP (ccf −0,30); no preço Petrobras a correlação é mais fraca e o máximo cai no horizonte 8. O desvio-padrão semanal do diesel ANP em 2026 é 0,55 R＄/L, cerca de 3,4 vezes o de 2025 (0,16 R＄/L), enquanto 2024 ficou quase estável (0,043 R＄/L). Esses picos de ccf descrevem o treino. O modelo usa o PPI da **mesma** semana do preço, no treino e no teste.

![Preço semanal ANP e Petrobras](../figures/eda/eda_preco.svg)

*Figura 6 — Preço semanal ANP e preço de lista Petrobras, gasolina e diesel.*

![ACF e PACF do diesel ANP](../figures/eda/eda_preco_acf.svg)

*Figura 7 — ACF e PACF do diesel ANP no treino (237 semanas).*


## Vetor de atividade

O painel mensal de treino tem 51 a 52 pontos até abril de 2026. Nenhuma das séries de tráfego pesado, indústria, varejo, desemprego ou venda de diesel da ANP rejeita a raiz unitária no nível, e a primeira diferença de tráfego e de indústria também fica acima do limiar usual, com amostra curta; por isso o SARIMAX mensal entra com diferença e sazonalidade anual (`s=12`) quando a busca de ordem confirma o termo.

As Figuras 8 a 11 mostram as quatro séries mensais no recorte de treino e de teste. Tráfego pesado, indústria e varejo repetem um ciclo de cerca de doze meses, o que pede o termo sazonal e justifica o Prophet com sazonalidade anual. O desemprego anda mais devagar, e o SARIMAX (0,1,2)(1,0,0)[12] é o desenho que a busca devolve. A Figura 12 decompõe a venda mensal de diesel da ANP: o pico do terceiro trimestre aparece em 2022 e em 2023, e a média móvel de 12 meses sobe no recorte. Essa sazonalidade de volume é o motivo de o modelo mensal não ficar só na primeira diferença.

A correlação cruzada com a média mensal do PPI do diesel é fraca: −0,19 no tráfego pesado e −0,13 na indústria, ambas no lag 0, e o varejo tem máximo −0,09 no lag 5, enquanto o desemprego correlaciona +0,15 no contemporâneo. O modelo mensal usa o produto desvio × volume somado **no mesmo mês** do alvo, no treino e no teste.

A participação da Petrobras no volume de diesel da ANP sai de 68,2% no 1º trimestre de 2022 para 63,4% no 3º trimestre de 2025. O tráfego de agosto de 2026 fica fora do alinhamento porque é o único mês ABCR sem par no IBGE.

![Tráfego pesado ABCR](../figures/eda/eda_atividade_heavy.svg)

*Figura 8 — Tráfego pesado ABCR, índice mensal.*

![Indústria PIM-PF](../figures/eda/eda_atividade_pim.svg)

*Figura 9 — Produção industrial (PIM-PF), índice mensal.*

![Varejo PMC](../figures/eda/eda_atividade_pmc.svg)

*Figura 10 — Volume de vendas no varejo (PMC), índice mensal.*

![Desemprego PNAD](../figures/eda/eda_atividade_unemp.svg)

*Figura 11 — Taxa de desemprego (PNAD Contínua), pontos percentuais.*

![Venda mensal de diesel ANP e tendência de 12 meses](../figures/eda/eda_decomp_diesel_anp.svg)

*Figura 12 — Venda mensal de diesel ANP (m³) e tendência de 12 meses.*


## Vetor da companhia

Há 17 trimestres completos no treino (1T22 a 1T26) e um ponto de teste (2T26). Com menos de 20 observações o ADF não se aplica, de modo que a série trimestral permanece descritiva e o SARIMAX entra só como leitura de um passo à frente. O 2T26 publicado tem diesel 742 mbpd, gasolina 404 mbpd, receita de diesel R＄ 38.262 milhões e de gasolina R＄ 15.209 milhões, vendas RT&M R＄ 163.193 milhões, lucro bruto R＄ 26.130 milhões e margem 16,01%. A nota 4 do ITR inclui R＄ 9.737 milhões de subvenção ao diesel e R＄ 816 milhões à gasolina, fato sem paralelo no treino, e por isso o erro desse vetor vale como indicação.

A Figura 13 mostra volume, receita, lucro bruto e margem de RT&M. O volume de diesel e de gasolina anda em degrau; a receita e o lucro oscilam entre trimestres. A margem cai de 20,27% no 1T26 para 16,01% no 2T26, no mesmo trimestre da subvenção. O SARIMAX com `prod_pbr` contemporâneo aponta 23,99% nesse ponto, na direção oposta ao ITR: o produto do próprio trimestre não descreve a queda de margem ligada à nota 4.

![Volume Petrobras](../figures/eda/eda_companhia_volume.svg)

*Figura 13 — Volume Petrobras de diesel e gasolina, mil barris por dia, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Receita RT&M](../figures/eda/eda_companhia_receita.svg)

*Figura 13b — Receita RT&M, R＄ milhões, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Lucro bruto RT&M](../figures/eda/eda_companhia_lucro.svg)

*Figura 13c — Lucro bruto RT&M, R＄ milhões, 1º trimestre de 2022 ao 2º trimestre de 2026.*

![Margem bruta RT&M](../figures/eda/eda_companhia_margem.svg)

*Figura 13d — Margem bruta RT&M da Petrobras, 1º trimestre de 2022 ao 2º trimestre de 2026.*

Quadro 3 — Exógena no mesmo período do alvo (o ccf do treino só descreve; o modelo não defasa)

| Vetor | Alvo | Exógena | Pico de ccf no treino | Uso no modelo |
|---|---|---|---|---|
| Mercado | PETR4 | `prod` do dia | 0 pregões | mesmo dia, inclusive no teste |
| Preço | ANP gasolina | `prod` da semana | 1 semana | mesma semana, inclusive no teste |
| Preço | ANP diesel | `prod` da semana | 2 semanas | mesma semana, inclusive no teste |
| Atividade | tráfego, PIM, PMC, desemprego | produto somado no mês | 0 a 5 meses, |ccf| < 0,20 | mesmo mês, inclusive no teste |
| Companhia | volume e RT&M | produto PPI × volume do próprio trimestre | — | mesmo trimestre (ex.: 1º jan.–31 mar. = 1T) |


# **Modelos**

*(Incremento da Etapa 3 — 26/10/2026.)*

O modelo base desta etapa é a linha ingênua, que repete o último valor de treino, e o SARIMAX com ordem de `auto_arima` no mesmo recorte. A ordem do SARIMAX sai de `auto_arima` no treino. O vetor diário entra sem sazonalidade anual, salvo a ACF pedir a semana (`s=5`); preço usa `s=52` e atividade `s=12`. A exógena é o `prod` (ou `prod_pbr`) do mesmo período do alvo. O teste usa o produto observado nesse horizonte, salvo o TimesFM.

A LSTM é uma rede PyTorch só no vetor diário, com PETR4 e `prod` contemporâneo em escala 0–1 ajustada no treino. A janela de PETR4 é o histórico; o último passo de `prod` é o produto do mesmo pregão que se prevê. A grade cobre o tamanho da janela (21 ou 63 pregões), com taxa de aprendizado 10⁻³, 40 épocas, 32 unidades e lote 64, e a validação é o último bloco de 21 pregões do treino. O TimesFM 3.0 (`google/timesfm-3.0-pytorch`) roda nesse vetor, sem treino nesta base, com PETR4 e `prod` do treino como covariável passada; os pesos baixam na execução para o cache do Hugging Face. A série mensal e a trimestral são curtas para a LSTM e para o TimesFM.

O Prophet entra nos vetores de preço e atividade, com sazonalidade anual e o mesmo `prod` contemporâneo do SARIMAX. O vetor da companhia usa o SARIMAX de um passo com `prod_pbr` (PPI × volume Petrobras daquele trimestre) e diferenciação `d=1`, conferido com o 2T26 publicado. A linha ingênua entra na tabela como dummy da disciplina.

Os ajustes gravam métricas em `results/etapa3_metrics.json` e as previsões em `results/etapa3_forecasts.json`. Neste Windows o OpenBLAS interrompe o SARIMAX, o Prophet e o TimesFM, de modo que a LSTM e a linha ingênua rodam neste kernel e as demais famílias executam no MacBook M4 (`bash run_etapa3_mac.sh`). A célula seguinte lê o JSON já gravado, para o notebook mostrar a tabela sem reajustar.

No Mac (darwin, arm64) o `auto_arima` devolve PETR4 em (0,1,0), ANP gasolina em (1,0,0)(0,0,0)[52], lista Petrobras gasolina em (0,1,0)(1,0,0)[52] e as demais séries semanais em (0,1,0)(0,0,0)[52]; no painel mensal o termo sazonal entra com `D=1` ou `P=1` e `s=12`. O TimesFM 3.0 roda no backend MLX. No 2T26 o SARIMAX entra com `prod_pbr` contemporâneo e `d=1` (grade statsmodels neste Windows): diesel (2,1,2), gasolina (0,1,2), receita (0,1,2), lucro bruto (0,1,2) e margem (0,1,2). As métricas de cada ajuste — MAE, MSE, RMSE e o gráfico real × previsto — ficam na seção Resultados; a discussão do melhor modelo fica para a Etapa 4.


In [6]:
# Tabela gravada em results/etapa3_metrics.json.
# Execute este bloco no kernel proj-iv. Ele só lê o JSON e mostra a tabela.
# Neste Windows o OpenBLAS aborta numpy.linalg se o ajuste rodar de novo.
#
# Para reajustar, fora do notebook, no diretório do projeto:
#   bash run_etapa3_mac.sh                      # MacBook M4: recorte congelado
#   python src/etapa3_models.py --only naive,lstm  # este Windows
import json

from IPython.display import display
from paths import forecasts_path, metrics_path

mp = metrics_path(project_root)
fp = forecasts_path(project_root)
payload = json.loads(mp.read_text(encoding="utf-8"))
tab = pd.DataFrame(payload["rows"])
tab = tab[tab["model"].isin(["naive", "sarimax", "prophet", "lstm", "timesfm_cov"])].copy()
prefer = ["vector", "target", "model", "horizon", "MAE", "MSE", "RMSE", "order", "window", "note"]
cols = [c for c in prefer if c in tab.columns] + [c for c in tab.columns if c not in prefer]
display(tab[cols])
print("métricas em", mp)
print("previsões em", fp, "existe" if fp.exists() else "ausente")


,vector,target,model,horizon,MAE,MSE,RMSE,order,window,note,seasonal_order,aic
0,mercado,PETR4,naive,5d,1.364000,1.946680,1.395235,NaN,NaN,NaN,NaN,NaN
1,mercado,PETR4,naive,21d,3.744286,18.606300,4.313502,NaN,NaN,NaN,NaN,NaN
2,preço,ANP gasolina,naive,4w,0.094387,0.010301,0.101495,NaN,NaN,NaN,NaN,NaN
3,preço,ANP gasolina,naive,8w,0.121302,0.022146,0.148815,NaN,NaN,NaN,NaN,NaN
4,preço,ANP diesel,naive,4w,0.398800,0.299569,0.547329,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
61,companhia,margem RT&M,sarimax,2Q26,7.976966,63.631984,7.976966,"[0, 1, 2]",NaN,prod_pbr contemporâneo (PPI × volume do própri...,None,70.613918
62,mercado,PETR4,lstm,5d,1.254363,2.185182,1.478236,NaN,21.0,NaN,NaN,NaN
63,mercado,PETR4,lstm,21d,4.943810,33.279070,5.768801,NaN,21.0,NaN,NaN,NaN
64,mercado,PETR4,timesfm_cov,5d,1.355001,1.918122,1.384963,NaN,NaN,NaN,NaN,NaN


métricas em D:\Users\Ricardo\Documents\opencode\proj_IV\results\etapa3_metrics.json
previsões em D:\Users\Ricardo\Documents\opencode\proj_IV\results\etapa3_forecasts.json existe


# **Resultados**

*(Incremento da Etapa 3 — 26/10/2026.)*

O modelo base desta etapa é a linha ingênua e o SARIMAX com `prod` contemporâneo, no mesmo treino e no mesmo teste de cada vetor. A LSTM e o TimesFM 3.0 entram no vetor diário com `prod`; o Prophet entra no preço e na atividade. Os Quadros 4 a 7 reúnem MAE, MSE e RMSE; o horizonte curto é prefixo do horizonte longo. Os erros saem da passagem M4 com `prod` (LSTM janela 21, TimesFM MLX). A leitura crítica do melhor resultado fica na Discussão da Etapa 4.

No teste de mercado (20 de agosto a 18 de setembro de 2026) PETR4 abre a R＄ 44,32, cai a R＄ 41,35 e fecha a R＄ 48,50, enquanto a linha ingênua repete o último fechamento de treino, R＄ 43,11. O SARIMAX (0,1,0) com `prod` do dia fica em RMSE 1,41 R＄ em cinco pregões e 4,21 R＄ em 21, a cerca de R＄ 0,10 da linha ingênua no horizonte longo. A LSTM com `prod` (janela 21) tem MAE 1,25 R＄ em cinco dias e RMSE 5,77 R＄ em 21. O TimesFM com `prod` do treino fica em RMSE 1,38 R＄ e 4,67 R＄. A Figura 14 mostra essa descida e essa subida: as famílias permanecem perto do último fechamento de treino.

Quadro 4 — Mercado (PETR4, R＄)

| Modelo | Horizonte | MAE | MSE | RMSE |
|---|---|---:|---:|---:|
| naive | 5d | 1,36 | 1,95 | 1,40 |
| naive | 21d | 3,74 | 18,61 | 4,31 |
| sarimax (0,1,0) | 5d | 1,38 | 2,00 | 1,41 |
| sarimax (0,1,0) | 21d | 3,66 | 17,72 | 4,21 |
| lstm + `prod` (janela 21) | 5d | 1,25 | 2,19 | 1,48 |
| lstm + `prod` (janela 21) | 21d | 4,94 | 33,28 | 5,77 |
| timesfm + `prod` | 5d | 1,36 | 1,92 | 1,38 |
| timesfm + `prod` | 21d | 4,03 | 21,79 | 4,67 |

![PETR4 real × previsto no teste](../figures/forecasts/fc_mercado_petr4.svg)

*Figura 14 — PETR4 no teste, 20 de agosto a 18 de setembro de 2026, real e previsto.*

O preço de lista da Petrobras permanece quase constante nas oito semanas de teste, de modo que a linha ingênua fica abaixo de 0,001 R＄/L. Com o `prod` da semana, o SARIMAX (0,1,0) se afasta desse degrau: RMSE 0,045 R＄/L na gasolina e 0,123 R＄/L no diesel em quatro semanas. O Prophet, com sazonalidade anual, afasta-se 0,42 a 0,56 R＄/L. No diesel ANP o SARIMAX reduz o RMSE de 0,55 para 0,45 R＄/L em quatro semanas. Na gasolina ANP o `auto_arima` devolve (1,0,0) sem diferença e o SARIMAX explode (RMSE 3,55 R＄/L em quatro semanas, contra 0,10 da naive), porque o `prod` está em R＄/dia e o preço em R＄/L. As Figuras 15 a 18 mostram esses recortes.

Quadro 5 — Preço (R＄/L)

| Alvo | Horizonte | MAE naive | RMSE naive | MAE SARIMAX | RMSE SARIMAX | MAE Prophet | RMSE Prophet |
|---|---|---:|---:|---:|---:|---:|---:|
| ANP gasolina | 4w | 0,094 | 0,101 | 3,45 | 3,55 | 0,466 | 0,471 |
| ANP gasolina | 8w | 0,121 | 0,149 | 4,68 | 4,92 | 0,512 | 0,541 |
| ANP diesel | 4w | 0,399 | 0,547 | 0,359 | 0,450 | 0,421 | 0,555 |
| ANP diesel | 8w | 0,230 | 0,391 | 0,255 | 0,337 | 0,760 | 0,873 |
| Petrobras gasolina | 4w | 0,0004 | 0,0004 | 0,041 | 0,045 | 0,418 | 0,419 |
| Petrobras gasolina | 8w | 0,0004 | 0,0004 | 0,053 | 0,056 | 0,417 | 0,418 |
| Petrobras diesel | 4w | 0,0001 | 0,0002 | 0,117 | 0,123 | 0,563 | 0,563 |
| Petrobras diesel | 8w | 0,0002 | 0,0003 | 0,158 | 0,166 | 0,454 | 0,469 |

![ANP diesel real × previsto](../figures/forecasts/fc_preco_anp_die.svg)

*Figura 15 — Preço ANP do diesel no teste, real e previsto.*

![ANP gasolina real × previsto](../figures/forecasts/fc_preco_anp_gas.svg)

*Figura 16 — Preço ANP da gasolina no teste, real e previsto.*

![Petrobras gasolina real × previsto](../figures/forecasts/fc_preco_pbr_gas.svg)

*Figura 17 — Preço de lista Petrobras da gasolina no teste, real e previsto.*

![Petrobras diesel real × previsto](../figures/forecasts/fc_preco_pbr_die.svg)

*Figura 18 — Preço de lista Petrobras do diesel no teste, real e previsto.*

No painel mensal o `prod` do mês e o termo sazonal reduzem o erro da linha ingênua no tráfego, na indústria e no varejo. No tráfego pesado o RMSE de três meses cai de 10,31 (naive) para 8,89 (SARIMAX) e 4,37 (Prophet), e na indústria de 6,77 para 2,38 e 1,69. No varejo o Prophet fica em RMSE 0,88 no horizonte de três meses, contra 2,72 da linha ingênua, e no desemprego o SARIMAX (0,1,2)(1,0,0)[12] tem RMSE 0,19 p.p. em três meses. As Figuras 19 a 22 mostram esses quatro alvos socioeconômicos lidos contra o `prod` do mês, com três pontos de teste.

Quadro 6 — Atividade

| Alvo | Horizonte | MAE naive | RMSE naive | MAE SARIMAX | RMSE SARIMAX | MAE Prophet | RMSE Prophet |
|---|---|---:|---:|---:|---:|---:|---:|
| tráfego pesado | 1m | 15,30 | 15,30 | 3,68 | 3,68 | 2,34 | 2,34 |
| tráfego pesado | 3m | 9,43 | 10,31 | 6,54 | 8,89 | 3,57 | 4,37 |
| indústria | 1m | 5,35 | 5,35 | 2,24 | 2,24 | 2,07 | 2,07 |
| indústria | 3m | 6,26 | 6,77 | 2,12 | 2,38 | 1,67 | 1,69 |
| varejo | 1m | 3,67 | 3,67 | 3,50 | 3,50 | 0,61 | 0,61 |
| varejo | 3m | 2,28 | 2,72 | 2,11 | 2,48 | 0,84 | 0,88 |
| desemprego | 1m | 0,20 | 0,20 | 0,13 | 0,13 | 0,29 | 0,29 |
| desemprego | 3m | 0,37 | 0,39 | 0,19 | 0,19 | 0,39 | 0,40 |

![Tráfego pesado real × previsto](../figures/forecasts/fc_atividade_abcr_heavy.svg)

*Figura 19 — Tráfego pesado ABCR no teste, real e previsto.*

![Indústria real × previsto](../figures/forecasts/fc_atividade_pim.svg)

*Figura 20 — Produção industrial (PIM-PF) no teste, real e previsto.*

![Varejo real × previsto](../figures/forecasts/fc_atividade_pmc.svg)

*Figura 21 — Volume de vendas no varejo (PMC) no teste, real e previsto.*

![Desemprego real × previsto](../figures/forecasts/fc_atividade_unemp.svg)

*Figura 22 — Taxa de desemprego (PNAD) no teste, real e previsto.*

A receita RT&M publicada no 2T26 é R＄ 163.193 milhões, R＄ 46.015 milhões acima do 1T26, e a margem cai de 20,27% para 16,01%, enquanto o volume de diesel sobe de 738 para 742 mbpd e o de gasolina cai de 413 para 404 mbpd. A nota 4 do ITR registra R＄ 9.737 milhões de subvenção ao diesel e R＄ 816 milhões à gasolina, fato sem paralelo no treino. O SARIMAX recebe o produto trimestral `prod_pbr` (desvio do PPI vezes o volume Petrobras do próprio trimestre) e diferencia com `d=1`. No 2T26 essa exógena soma abril a junho de 2026. Com o regressor, a receita prevista sobe de R＄ 117.178 milhões (naive) para R＄ 129.165 milhões, ainda R＄ 34.028 milhões abaixo do publicado. Diesel, lucro bruto e margem ficam mais longe da linha ingênua: o modelo empurra a margem para 23,99% enquanto o ITR cai a 16,01%. Com um ponto de teste e a subvenção, o Quadro 7 permanece indicação. As Figuras 23 a 27 comparam o real com a naive e o SARIMAX no recorte 1T26–2T26, no mesmo formato das Figuras 19 a 22.

Quadro 7 — Companhia, 2T26 publicado contra o último trimestre de treino (1T26)

| Alvo | 1T26 (naive) | SARIMAX | 2T26 publicado | MAE naive | MAE SARIMAX | ordem |
|---|---:|---:|---:|---:|---:|---|
| diesel mbpd | 738 | 785 | 742 | 4 | 42,91 | (2,1,2) |
| gasolina mbpd | 413 | 395 | 404 | 9 | 9,11 | (0,1,2) |
| receita RT&M (R＄ milhões) | 117.178 | 129.165 | 163.193 | 46.015 | 34.028 | (0,1,2) |
| lucro bruto RT&M (R＄ milhões) | 23.750 | 28.586 | 26.130 | 2.380 | 2.456 | (0,1,2) |
| margem RT&M (%) | 20,27 | 23,99 | 16,01 | 4,26 | 7,98 | (0,1,2) |

![Diesel mbpd real × previsto](../figures/forecasts/fc_companhia_diesel_mbpd.svg)

*Figura 23 — Diesel Petrobras (mbpd) no teste, real e previsto.*

![Gasolina mbpd real × previsto](../figures/forecasts/fc_companhia_gasoline_mbpd.svg)

*Figura 24 — Gasolina Petrobras (mbpd) no teste, real e previsto.*

![Receita RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_sales_revenue_rsmn.svg)

*Figura 25 — Receita RT&M no teste, real e previsto.*

![Lucro bruto RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_gross_profit_rsmn.svg)

*Figura 26 — Lucro bruto RT&M no teste, real e previsto.*

![Margem RT&M real × previsto](../figures/forecasts/fc_companhia_rtm_gross_margin_pct.svg)

*Figura 27 — Margem RT&M no teste, real e previsto.*


# **Discussão e Conclusão**

*(Etapa 4.)* Qualidades e limites do trabalho. Alcance do objetivo. Melhorias possíveis.


# **Apresentação**

*(Etapa 4.)* Link do vídeo (apresentação com slides).


# **Referências**

ASSOCIAÇÃO BRASILEIRA DE CONCESSIONÁRIAS DE RODOVIAS. *Índice ABCR*. São Paulo: ABCR, 2026. Disponível em: https://melhoresrodovias.org.br/. Acesso em: 19 ago. 2026.

ASSOCIAÇÃO BRASILEIRA DOS IMPORTADORES DE COMBUSTÍVEIS. *PPI*. Rio de Janeiro: ABICOM, 2026. Disponível em: https://abicom.com.br/categoria/ppi/. Acesso em: 19 ago. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Preços de produtores e importadores de derivados de petróleo e biodiesel*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/assuntos/precos-e-defesa-da-concorrencia/precos/precos-de-produtores-e-importadores-de-derivados-de-petroleo-e-biodiesel. Acesso em: 27 set. 2026.

AGÊNCIA NACIONAL DO PETRÓLEO, GÁS NATURAL E BIOCOMBUSTÍVEIS. *Vendas de combustíveis (m³)*. Brasília: ANP, 2026. Disponível em: https://www.gov.br/anp/pt-br/centrais-de-conteudo/dados-estatisticos. Acesso em: 19 ago. 2026.

BANCO CENTRAL DO BRASIL. *Sistema Gerenciador de Séries Temporais (SGS)*. Brasília: BCB, 2026. Disponível em: https://www3.bcb.gov.br/sgspub/. Acesso em: 19 ago. 2026.

B3 S.A. – BRASIL, BOLSA, BALCÃO. *Cotações históricas*. São Paulo: B3, 2026. Disponível em: https://www.b3.com.br/pt_br/market-data-e-indices/servicos-de-dados/market-data/historico/mercado-a-vista/cotacoes-historicas/. Acesso em: 19 ago. 2026.

DAS, A.; KONG, W.; SEN, R.; ZHOU, Y. A decoder-only foundation model for time-series forecasting. In: INTERNATIONAL CONFERENCE ON MACHINE LEARNING, 41., 2024, Viena. *Proceedings* [...]. [S. l.]: PMLR, 2024. v. 235. Disponível em: https://arxiv.org/abs/2310.10688. Acesso em: 25 set. 2026.

GOOGLE RESEARCH. *TimesFM*: time series foundation model. [S. l.]: GitHub, 2026. Disponível em: https://github.com/google-research/timesfm. Acesso em: 25 set. 2026.

HYNDMAN, R. J.; ATHANASOPOULOS, G. *Forecasting*: principles and practice. 3. ed. Melbourne: OTexts, 2021. Disponível em: https://otexts.com/fpp3/. Acesso em: 24 ago. 2026.

INSTITUTO BRASILEIRO DE GEOGRAFIA E ESTATÍSTICA. *SIDRA*: Sistema IBGE de Recuperação Automática. Rio de Janeiro: IBGE, 2026. Disponível em: https://sidra.ibge.gov.br/. Acesso em: 19 ago. 2026.

LOPES, A. O.; VIEGAS, T. O. C.; CONTE FILHO, C. G.; CARVALHO, V. S. Fatores influenciando preços de ações Petrobras (PETR4) entre 2009 e 2020. *Research, Society and Development*, v. 10, n. 7, 2021. Disponível em: https://rsdjournal.org/index.php/rsd/article/view/16294. Acesso em: 25 set. 2026.

ORGANIZAÇÃO DAS NAÇÕES UNIDAS. *Sustainable Development Goals*. Nova York: ONU, 2024. Disponível em: https://www.undp.org/sustainable-development-goals. Acesso em: 24 ago. 2026.

PETROBRAS. *Central de resultados*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://www.investidorpetrobras.com.br/resultados-e-comunicados/central-de-resultados/. Acesso em: 19 ago. 2026.

PETROBRAS. *Tabelas de preços de venda às distribuidoras*. Rio de Janeiro: Petrobras, 2026. Disponível em: https://precos.petrobras.com.br. Acesso em: 19 ago. 2026.

PUENTES MÁRQUEZ, J.; OLIVEIRA RIBEIRO, C. de; RUELAS SANTOYO, E.; FIGUEROA FERNÁNDEZ, V. Ethanol fuel demand forecasting in Brazil using a LSTM recurrent neural network approach. *IEEE Latin America Transactions*, v. 19, n. 4, p. 551-558, 2021. Disponível em: https://latamt.ieeer9.org/index.php/transactions/article/view/3713. Acesso em: 25 set. 2026.

SERRANO, A. L. M. et al. Forecasting ethanol and gasoline consumption in Brazil: advanced temporal models for sustainable energy management. *Energies*, v. 18, n. 6, 1501, 2025. DOI: https://doi.org/10.3390/en18061501.

SILVA, L. A.; PERES, S. M.; BOSCARIOLI, C. *Introdução à mineração de dados com aplicações em R*. 1. ed. São Paulo: LTC, 2016.

TAYLOR, S. J.; LETHAM, B. Forecasting at scale. *The American Statistician*, v. 72, n. 1, p. 37-45, 2018. DOI: https://doi.org/10.1080/00031305.2017.1380080.